In [0]:
%pip install osmium

In [0]:
import os, shutil, datetime, json
import numpy as np
import pandas as pd
import requests
import osmium
import shapely
from shapely.geometry import shape
from pyspark.sql import functions as F

CATALOG= "hawkerreach"
from dq import DQLog  # dq.py at the repo root
dq= DQLog(spark, "03_dist_matrix")

In [0]:
PBF_URL= "https://download.geofabrik.de/asia/malaysia-singapore-brunei-latest.osm.pbf"
PBF_TMP= "/tmp/msb.osm.pbf"
PBF_VOL= f"/Volumes/{CATALOG}/bronze/raw_files/osm_malaysia-singapore-brunei_{datetime.date.today()}.osm.pbf"

if not os.path.exists(PBF_TMP):
    with requests.get(PBF_URL, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(PBF_TMP, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 << 20):
                f.write(chunk)
if not os.path.exists(PBF_VOL):
    shutil.copy(PBF_TMP, PBF_VOL)  # dated raw copy = reproducible bronze source
print("size MB:", round(os.path.getsize(PBF_TMP) / 1e6), "| saved as:", PBF_VOL)

In [0]:
WALK= {"footway", "path", "pedestrian", "steps", "residential", "living_street", "service", "unclassified",
       "tertiary", "tertiary_link", "secondary", "secondary_link", "primary", "primary_link",
       "track", "corridor", "cycleway", "road"}  # motorways + expressways excluded: no walking there
LON0, LON1, LAT0, LAT1= 103.55, 104.15, 1.15, 1.50  # rough Singapore box; exact clip below

class WalkWays(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.rows= []
    def way(self, w):
        t= w.tags
        hw= t.get("highway")
        if hw not in WALK or t.get("foot") == "no" or t.get("area") == "yes":
            return
        if t.get("access") in ("private", "no") and t.get("foot") not in ("yes", "designated", "permissive"):
            return
        pts= [(n.ref, n.location.lon, n.location.lat) for n in w.nodes if n.location.valid()]
        if not any(LON0 <= x <= LON1 and LAT0 <= y <= LAT1 for _, x, y in pts):
            return
        for (a, ax, ay), (b, bx, by) in zip(pts, pts[1:]):
            self.rows.append((a, ax, ay, b, bx, by, hw))

h= WalkWays()
h.apply_file(PBF_TMP, locations=True, idx="flex_mem")
seg= pd.DataFrame(h.rows, columns=["u", "ux", "uy", "v", "vx", "vy", "highway"])

# exact clip: keep segments with both ends inside Singapore's subzone boundaries (drops Johor Bahru)
sg= shapely.union_all([shape(json.loads(g)) for g in spark.table(f"{CATALOG}.silver.subzones").select("geometry").toPandas()["geometry"]])
shapely.prepare(sg)
inside= shapely.contains_xy(sg, seg["ux"], seg["uy"]) & shapely.contains_xy(sg, seg["vx"], seg["vy"])
n_seg= len(seg)
seg= seg[inside].copy()

# segment length in metres (haversine)
R= 6371000
lat1, lat2= np.radians(seg["uy"]), np.radians(seg["vy"])
dlat, dlon= lat2 - lat1, np.radians(seg["vx"] - seg["ux"])
seg["length_m"]= 2 * R * np.arcsin(np.sqrt(np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2))

nodes= pd.concat([seg[["u", "ux", "uy"]].set_axis(["node_id", "lon", "lat"], axis=1),
                  seg[["v", "vx", "vy"]].set_axis(["node_id", "lon", "lat"], axis=1)]).drop_duplicates("node_id")
edges= seg[["u", "v", "length_m", "highway"]]

spark.createDataFrame(nodes).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.walk_nodes")
spark.createDataFrame(edges).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.walk_edges")
print("segments:", len(edges), "| nodes:", len(nodes), "| total km:", round(edges["length_m"].sum() / 1000))
dq.step("walk_edges", n_seg, len(edges), "OSM walkable segments kept inside Singapore")
print(edges["highway"].value_counts().head(8).to_string())

In [0]:
import networkx as nx
from scipy.spatial import cKDTree
from collections import defaultdict

nodes= spark.table(f"{CATALOG}.silver.walk_nodes").toPandas()
edges= spark.table(f"{CATALOG}.silver.walk_edges").toPandas()

G= nx.Graph()
e= edges.sort_values("length_m", ascending=False)  # duplicate segments: shortest is added last and wins
G.add_weighted_edges_from(zip(e["u"], e["v"], e["length_m"]), weight="length_m")
lcc= max(nx.connected_components(G), key=len)
print("nodes:", G.number_of_nodes(), "| largest connected piece:", len(lcc), f"({len(lcc) / G.number_of_nodes():.1%})")
G= G.subgraph(lcc).copy()

# snap points to nearest graph node (flat metres projection, fine at Singapore's scale)
LAT0= 1.35
def xy(lon, lat):
    return np.column_stack([np.asarray(lon) * 111320 * np.cos(np.radians(LAT0)), np.asarray(lat) * 110574])

nd= nodes[nodes["node_id"].isin(list(lcc))].reset_index(drop=True)
tree= cKDTree(xy(nd["lon"], nd["lat"]))

def snap(df):
    d, i= tree.query(xy(df["lon"], df["lat"]))
    return df.assign(node_id=nd["node_id"].to_numpy()[i], snap_m=d)

blk= snap(spark.table(f"{CATALOG}.silver.blocks").select("block_id", "lon", "lat").toPandas())
cen= snap(spark.table(f"{CATALOG}.silver.centres").select("centre_id", "lon", "lat").toPandas())
print(f"block snap m | median {blk['snap_m'].median():.0f} | max {blk['snap_m'].max():.0f} | over 150 m: {(blk['snap_m'] > 150).sum()}")
print(f"centre snap m | median {cen['snap_m'].median():.0f} | max {cen['snap_m'].max():.0f}")

# one shortest-path search per centre, 2 km cutoff
CUTOFF= 2000
node_blocks= defaultdict(list)
for b, n, s in zip(blk["block_id"], blk["node_id"], blk["snap_m"]):
    node_blocks[n].append((b, s))

rows= []
for c, n, cs in zip(cen["centre_id"], cen["node_id"], cen["snap_m"]):
    dist= nx.single_source_dijkstra_path_length(G, n, cutoff=CUTOFF, weight="length_m")
    for node, d in dist.items():
        for b, bs in node_blocks.get(node, ()):
            w= d + bs + cs
            if w <= CUTOFF:
                rows.append((b, c, w))
dm= pd.DataFrame(rows, columns=["block_id", "centre_id", "walk_m"])

spark.createDataFrame(dm).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.dist_matrix")

# checks
nearest= dm.groupby("block_id")["walk_m"].min()
print("pairs:", len(dm), "| blocks with a centre within 2 km:", nearest.size, "of", len(blk))
dq.step("dist_matrix", len(blk), nearest.size, "blocks with a centre within 2 km")
print(f"nearest centre walk m | median {nearest.median():.0f} | 90th pct {nearest.quantile(0.9):.0f}")
chk= dm.merge(blk[["block_id", "lon", "lat"]], on="block_id").merge(cen[["centre_id", "lon", "lat"]], on="centre_id", suffixes=("_b", "_c"))
straight= np.hypot(*(xy(chk["lon_b"], chk["lat_b"]) - xy(chk["lon_c"], chk["lat_c"])).T)
print(f"walk / straight-line ratio | median {np.median(chk['walk_m'] / straight):.2f} | min {np.min(chk['walk_m'] / straight):.2f}")
no_centre= spark.table(f"{CATALOG}.silver.blocks").join(spark.createDataFrame(nearest.reset_index()[["block_id"]]), "block_id", "left_anti")
display(no_centre.groupBy("pln_area_n").agg(F.count("*").alias("blocks"), F.round(F.sum("est_seniors")).alias("seniors")).orderBy(F.desc("blocks")))